## Quality of Care Report

This report displays a compact year-level summary of quality-of-care indicators and points to generated map outputs.

In [ ]:
ROOT_PATH <- "~/workspace"
source(file.path(ROOT_PATH, "pipelines", "snt_dhis2_quality_of_care", "utils", "snt_dhis2_quality_of_care_report.r"))

snt_environment <- get_setup_variables(SNT_ROOT_PATH = ROOT_PATH, packages = c("jsonlite", "data.table", "arrow", "sf", "ggplot2", "glue", "reticulate", "RColorBrewer", "dplyr", "knitr", "scales", "gridExtra", "IRdisplay"))
config_json <- load_snt_config(snt_environment$CONFIG_PATH, "SNT_config.json")
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
DHIS2_FORMATTED_DATASET <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED
PIPELINE_PATH <- file.path(snt_environment$PIPELINES_PATH, "snt_dhis2_quality_of_care")
OUTPUT_DATA_PATH <- file.path(snt_environment$DATA_PATH, "dhis2", "quality_of_care")
REPORT_OUTPUTS_PATH <- file.path(PIPELINE_PATH, "reporting", "outputs")
FIGURES_PATH <- file.path(REPORT_OUTPUTS_PATH, "figures")
dir.create(OUTPUT_DATA_PATH, recursive = TRUE, showWarnings = FALSE)
dir.create(REPORT_OUTPUTS_PATH, recursive = TRUE, showWarnings = FALSE)
dir.create(FIGURES_PATH, recursive = TRUE, showWarnings = FALSE)

In [ ]:
# Load latest district-year output and build summary
qoc_ctx <- load_latest_quality_of_care_output(OUTPUT_DATA_PATH, COUNTRY_CODE)
qoc <- qoc_ctx$qoc
latest_file <- qoc_ctx$latest_file

summary_tbl <- build_quality_of_care_summary(qoc)
summary_paths <- save_quality_of_care_summary_outputs(
  summary_tbl = summary_tbl,
  report_outputs_path = REPORT_OUTPUTS_PATH,
  country_code = COUNTRY_CODE
)

knitr::kable(summary_tbl, caption = "Quality of Care - Year-level summary")

cat(glue::glue("\nLoaded file: {latest_file}\n"))
cat(glue::glue("Map outputs folder: {FIGURES_PATH}\n"))
cat(glue::glue("Summary data saved to: {summary_paths$summary_parquet}, {summary_paths$summary_csv}\n"))

## Graphs by Year

In [ ]:
# Build, save, and display year-level indicator chart
charts_file <- save_quality_of_care_summary_charts(
  summary_tbl = summary_tbl,
  figures_path = FIGURES_PATH,
  country_code = COUNTRY_CODE
)

IRdisplay::display_png(file = normalizePath(path.expand(charts_file)))

## Maps by District and Year

Maps are generated directly from the quality-of-care data and district shapes.

In [ ]:
# Load shapes, regenerate yearly maps, and display them
shapes_filename <- glue::glue("{COUNTRY_CODE}_shapes.geojson")
shapes <- load_dataset_file(
  dataset_id = DHIS2_FORMATTED_DATASET,
  filename = shapes_filename
)

save_quality_of_care_maps(
  qoc_dt = qoc,
  shapes_sf = shapes,
  figures_path = FIGURES_PATH
)

years <- sort(unique(qoc$YEAR))
years_regex <- paste(years, collapse = "|")
map_files <- list.files(
  FIGURES_PATH,
  pattern = glue::glue("^(testing_rate|treatment_rate|case_fatality_rate|prop_adm_malaria|prop_malaria_deaths|allout|presumed_cases)_({years_regex})[.]png$"),
  full.names = TRUE
)
map_files <- sort(map_files)

for (map_file in map_files) {
  IRdisplay::display_png(file = normalizePath(path.expand(map_file)))
}